# SWYNEX Task 1 — Data Cleaning & Preparation

**Dataset:** Customer Personality Analysis / Marketing Campaign  
**Tools:** Python, Pandas, NumPy, Jupyter Notebook

This notebook profiles a raw customer dataset, identifies data-quality issues, applies cleaning steps, validates the result, and exports a cleaned CSV.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../data/raw/marketing_campaign.csv')
OUTPUT_PATH = Path('../data/cleaned/customer_personality_cleaned.csv')

df = pd.read_csv(RAW_PATH, sep=None, engine='python')
df.shape

## 1. Initial data inspection

In [ ]:
display(df.head())
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('\nData types:')
display(df.dtypes.to_frame('dtype'))

## 2. Missing values and duplicate checks

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)
display(missing[missing['missing_count'] > 0])
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate customer IDs:', df['ID'].duplicated().sum())

## 3. Check categorical consistency

In [ ]:
print('Education values:')
print(df['Education'].value_counts(dropna=False))
print('\nMarital Status values:')
print(df['Marital_Status'].value_counts(dropna=False))

## 4. Cleaning decisions

- Standardize column names to lowercase snake_case.
- Remove exact duplicate rows and duplicate customer IDs.
- Convert `Dt_Customer` to a proper datetime field.
- Standardize `2n Cycle` to `2nd Cycle`.
- Group uncommon marital-status labels (`Alone`, `Absurd`, `YOLO`) into `Other`.
- Treat birth years before 1900 as implausible values and impute them with the median year.
- Fill missing `Income` values with the median income.
- Convert count/indicator columns to integer types where appropriate.

In [ ]:
clean = df.copy()

clean.columns = (clean.columns.str.strip().str.lower()
                 .str.replace(r'[^a-z0-9]+', '_', regex=True)
                 .str.strip('_'))

clean = clean.drop_duplicates()
clean = clean.drop_duplicates(subset=['id'], keep='first')
clean['dt_customer'] = pd.to_datetime(clean['dt_customer'], dayfirst=True, errors='coerce')
clean['education'] = clean['education'].astype('string').str.strip().replace({'2n Cycle': '2nd Cycle'})
clean['marital_status'] = clean['marital_status'].astype('string').str.strip().replace({'Alone': 'Other', 'Absurd': 'Other', 'YOLO': 'Other'})
clean.loc[clean['year_birth'] < 1900, 'year_birth'] = np.nan
clean['year_birth'] = clean['year_birth'].fillna(clean['year_birth'].median()).round().astype('Int64')
clean['income'] = clean['income'].fillna(clean['income'].median()).round(2)
clean.head()

## 5. Validation after cleaning

In [ ]:
print('Rows:', len(clean))
print('Columns:', len(clean.columns))
print('Missing values remaining:', int(clean.isna().sum().sum()))
print('Duplicate rows remaining:', int(clean.duplicated().sum()))
print('Duplicate IDs remaining:', int(clean['id'].duplicated().sum()))
print('\nData types after cleaning:')
display(clean.dtypes.to_frame('dtype'))

## 6. Export cleaned dataset

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_PATH, index=False)
print(f'Cleaned dataset saved to: {OUTPUT_PATH}')